# Model 3 — Multi-output MLP on frozen ModernBERT embeddings

Predicts the **6-vector of compression perplexities** from prompt text.

- Data pulled live from GitHub (`perplexity_wide_complete.csv`).
- Targets in **log space**; inverted to raw perplexity for reporting.
- **Stratified random 70/30 split by dataset**, `random_state=42`.
- Trained model saved to `artifacts/`.

Requires `kv_common.py` in the same folder.

## Colab setup

Run this cell **first**. It clones the repo (so `kv_common.py` is available),
installs packages, and optionally mounts Google Drive so your cached embeddings
and trained models survive a disconnect.

> For notebook 04 (LoRA), also enable a GPU: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# Clone the repo so kv_common.py and artifacts live in one place.
import os
if not os.path.exists("KVCacheCompression"):
    !git clone -q https://github.com/yoshikodes/KVCacheCompression.git
# Work inside the notebooks folder (edit if your notebooks live elsewhere).
if os.path.basename(os.getcwd()) != "notebooks":
    %cd KVCacheCompression/notebooks
print("cwd:", os.getcwd())
assert os.path.exists("kv_common.py"), "kv_common.py not found"


In [ ]:
# Install packages not preinstalled on Colab.
# If you get an import error right after this, do Runtime -> Restart, then re-run from the top.
!pip install -q sentence-transformers scikit-learn scipy requests joblib

In [ ]:
# Optional but recommended: mount Drive so artifacts/ persists across sessions.
# Without this, trained models and cached embeddings are lost on disconnect,
# and notebook 05 will only see models trained in the current session.
USE_DRIVE = True  # set False to keep everything ephemeral

import os
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ART_DIR = "/content/drive/MyDrive/kv_artifacts"
        os.makedirs(ART_DIR, exist_ok=True)
        # Link ./artifacts -> Drive so all the notebook's save paths persist.
        if os.path.islink("artifacts") or os.path.exists("artifacts"):
            if not os.path.islink("artifacts"):
                import shutil; shutil.rmtree("artifacts", ignore_errors=True)
        if not os.path.exists("artifacts"):
            os.symlink(ART_DIR, "artifacts")
        print("artifacts -> ", os.path.realpath("artifacts"))
    except Exception as e:
        print("Drive mount skipped:", e)
        os.makedirs("artifacts", exist_ok=True)
else:
    os.makedirs("artifacts", exist_ok=True)

In [ ]:
import numpy as np, pandas as pd, os, joblib
import kv_common as kv

os.makedirs("artifacts", exist_ok=True)
df = kv.load_data()
train_df, test_df = kv.stratified_split(df)

LOG_SPACE = True
y_train = kv.get_targets(train_df, log_space=LOG_SPACE)
y_test  = kv.get_targets(test_df,  log_space=LOG_SPACE)
baseline = kv.baseline_predict_mean(y_train, len(y_test))
baseline_metrics = kv.evaluate(y_test, baseline)
print("Mean-baseline OVERALL MAE_log:",
      round(baseline_metrics[baseline_metrics.setting=='OVERALL'].MAE_log.iloc[0], 4))


## Encode (reuses the cache from notebook 02 if present)

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

ENCODER_NAME = "nomic-ai/modernbert-embed-base"  # ModernBERT-based, 8192 ctx
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| encoder:", ENCODER_NAME)

encoder = SentenceTransformer(ENCODER_NAME, device=device)

# Cache embeddings to disk so you only encode once across notebooks.
CACHE = "artifacts/embeddings_modernbert.npz"
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    emb_train, emb_test = z["train"], z["test"]
    print("loaded cached embeddings", emb_train.shape, emb_test.shape)
else:
    emb_train = encoder.encode(train_df["prompt"].tolist(),
                               batch_size=32, show_progress_bar=True,
                               convert_to_numpy=True)
    emb_test  = encoder.encode(test_df["prompt"].tolist(),
                               batch_size=32, show_progress_bar=True,
                               convert_to_numpy=True)
    np.savez(CACHE, train=emb_train, test=emb_test)
    print("encoded + cached", emb_train.shape, emb_test.shape)

## MLP with a 6-output head

Set `USE_NLL = True` for a Gaussian negative-log-likelihood head that also
predicts a per-output uncertainty (sigma). `False` uses plain MSE.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(kv.RANDOM_STATE)
USE_NLL = False

# Standardize targets for stable training; invert at eval.
y_mean = y_train.mean(0); y_std = y_train.std(0) + 1e-8
ytr_n = (y_train - y_mean) / y_std
yte_n = (y_test  - y_mean) / y_std

Xtr = torch.tensor(emb_train, dtype=torch.float32)
Xte = torch.tensor(emb_test,  dtype=torch.float32)
Ytr = torch.tensor(ytr_n, dtype=torch.float32)

n_out = len(kv.SETTINGS)
class MLP(nn.Module):
    def __init__(self, d_in, n_out, nll=False):
        super().__init__()
        self.nll = nll
        self.trunk = nn.Sequential(
            nn.Linear(d_in,512), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(512,256), nn.GELU(), nn.Dropout(0.2))
        self.mu = nn.Linear(256, n_out)
        if nll: self.logvar = nn.Linear(256, n_out)
    def forward(self,x):
        h = self.trunk(x)
        if self.nll: return self.mu(h), self.logvar(h)
        return self.mu(h)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLP(Xtr.shape[1], n_out, USE_NLL).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
dl = DataLoader(TensorDataset(Xtr, Ytr), batch_size=64, shuffle=True)

def nll_loss(mu, logvar, y):
    return (0.5*torch.exp(-logvar)*(y-mu)**2 + 0.5*logvar).mean()

EPOCHS = 100
for ep in range(EPOCHS):
    model.train()
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        if USE_NLL:
            mu, lv = model(xb); loss = nll_loss(mu, lv, yb)
        else:
            loss = nn.functional.mse_loss(model(xb), yb)
        loss.backward(); opt.step()
    if (ep+1) % 20 == 0:
        print(f"epoch {ep+1}/{EPOCHS} loss {loss.item():.4f}")

## Evaluate

In [ ]:
model.eval()
with torch.no_grad():
    out = model(Xte.to(device))
    mu = (out[0] if USE_NLL else out).cpu().numpy()
pred = mu * y_std + y_mean  # invert standardization -> log-perplexity
metrics_mlp = kv.evaluate(y_test, pred)
print(metrics_mlp.round(4).to_string(index=False))
kv.print_comparison(metrics_mlp, baseline_metrics)

## Save

In [ ]:
torch.save({"state_dict": model.state_dict(),
            "y_mean": y_mean, "y_std": y_std, "use_nll": USE_NLL,
            "encoder": ENCODER_NAME, "d_in": int(Xtr.shape[1]),
            "settings": kv.SETTINGS, "log_space": LOG_SPACE},
           "artifacts/model3_mlp.pt")
metrics_mlp.to_csv("artifacts/model3_mlp_metrics.csv", index=False)
print("saved artifacts/model3_mlp.pt")